In [ ]:
"""
Archival Processing project planner formatted for google collab. 
Includes user-friendly layout 
Outputs chart and vizualization

"""
# @title
import datetime
import holidays
import math
import matplotlib.pyplot as plt

try:
    import holidays
except ImportError:
    !pip install holidays
    import holidays

def get_calendar_stats(total_working_days, staff_count, work_days_indices):
    """Calculates year/month/day breakdown based on the specific work schedule."""
    days_per_week = len(work_days_indices)
    days_in_year = 52 * days_per_week
    days_in_month = days_in_year / 12 if days_in_year > 0 else 0

    def breakdown(days):
        if days_in_year == 0: return 0, 0, 0
        years = int(days // days_in_year)
        remaining_after_years = days % days_in_year
        months = int(remaining_after_years // days_in_month)
        remaining_days = round(remaining_after_years % days_in_month, 2)
        return years, months, remaining_days

    total_stats = breakdown(total_working_days)
    days_per_person = total_working_days / staff_count if staff_count > 0 else 0
    pp_stats = breakdown(days_per_person)
    return total_stats, pp_stats, days_per_person

def get_completion_date(start_date, working_days, work_days_indices):
    """Steps through calendar, only counting scheduled workdays and skipping holidays."""
    us_holidays = holidays.US()
    current_date = start_date
    days_added = 0
    target_days = math.ceil(working_days)

    if start_date.weekday() in work_days_indices and start_date not in us_holidays:
        days_added = 1

    while days_added < target_days:
        current_date += datetime.timedelta(days=1)
        if current_date.weekday() in work_days_indices and current_date not in us_holidays:
            days_added += 1
    return current_date

# **Archival Processing Project Planner**
User friendly project planning tool for mapping out project timelines and staffing. Fill out the fields to determine project timelines.


*   **linear_feet**: Enter the total linear feet in the collection
*   **ami_recordings**: Enter the total number of AMI recordings in the collection
*   **digital_carriers**: Enter the total number of digital carriers in the colection. If you know the number of GB, enter "0"
*   **gigabytes**: Enter the total digital extent if known, otherwise use **digital_carriers**
*   **linear_ft_per_day**: Enter the processing rate for per day. The default is 1 linear foot per day
*   **ami_per_day**: Enter the processing rate for AMI per day. The default is 30 recordings.
*   **carriers_per_day**: Enter the number of digital carriers that can be processed in a day. The default is 1. 
*   **gb_per_day**: Enter the number of GB that can be processeed per day. The default is 0.2, e.g. 5 days per GB.
*   **staff_fte**: Enter the total number of staff working on the project. Part time staff can be entered as "0.5"
*  **hours_per_workday**: Enter the total number of hours in a workday. The default is 7
*   **start_date_input**: Enter the project start date. It will defaut to the current date. Holidays and weekends are not allowed.
*   **days of the week checkboxes** By default all boxes are checked. If the project will not occur every day, adjust as needed

When all fields are filled out correctly, press the play button. The script will calculate the total number of staff hours, workdays, and duration. It also produces a visualization that displays the time for linear feet, AMI, and electronic records.

In [ ]:
# @title Archival Project Planner Settings { run: "auto" }

# --- STEP 1: QUANTITIES ---
linear_feet = 30 # @param {type:"number"}
ami_recordings = 700 # @param {type:"number"}
digital_carriers = 100 # @param {type:"number"}
gigabytes = 0 # @param {type:"number"}

# --- STEP 2: PROCESSING RATES ---
linear_ft_per_day = 15 # @param {type:"number"}
ami_per_day = 100 # @param {type:"number"}
carriers_per_day = 1 # @param {type:"number"}  # Changed from days_per_carrier
days_per_gb = 5 # @param {type:"number"}

# --- STEP 3: STAFFING & SCHEDULE ---
staff_fte = 1 # @param {type:"number"}
hours_per_workday = 7.0 # @param {type:"number"}
start_date_input = "2026-07-23" # @param {type:"date"}

# --- STEP 4: WORK DAYS SELECTION ---
Monday = True # @param {type:"boolean"}
Tuesday = True # @param {type:"boolean"}
Wednesday = True # @param {type:"boolean"}
Thursday = True # @param {type:"boolean"}
Friday = True # @param {type:"boolean"}

# Convert checkboxes to indices (0=Mon, 4=Fri)
work_days_indices = []
days_labels = ["Mon", "Tue", "Wed", "Thu", "Fri"]
selected_days_text = []
for i, day in enumerate([Monday, Tuesday, Wednesday, Thursday, Friday]):
    if day:
        work_days_indices.append(i)
        selected_days_text.append(days_labels[i])

# --- CALCULATIONS ---
start_date = datetime.datetime.strptime(start_date_input, "%Y-%m-%d").date()

# Effort breakdown
efforts = {
    'Physical': (linear_feet / linear_ft_per_day) if linear_feet > 0 else 0,
    'AMI': (ami_recordings / ami_per_day) if ami_recordings > 0 else 0,
    'Digital Carriers': (digital_carriers / carriers_per_day) if digital_carriers > 0 and carriers_per_day > 0 else 0, 
    'Digital (GB)': (gigabytes * gb_per_day) if gigabytes > 0 else 0
}

total_working_days = sum(efforts.values())

if not work_days_indices:
    print(" ERROR: Please select at least one workday in the checkboxes!")
elif total_working_days == 0:
    print("No work days entered. Please enter quantities in Step 1.")
else:
    # Get stats and end date
    total_stats, pp_stats, calendar_days = get_calendar_stats(total_working_days, staff_fte, work_days_indices)
    end_date = get_completion_date(start_date, calendar_days, work_days_indices)
    total_hours = total_working_days * hours_per_workday

    # --- PRINT REPORT ---
    print(f"{'='*50}")
    print(f"ESTIMATED COMPLETION: {end_date.strftime('%A, %B %d, %Y')}")
    print(f"{'-'*50}")
    print(f"Team Size:         {staff_fte} Staff (FTE)")
    print(f"Schedule:          {', '.join(selected_days_text)} ({hours_per_workday} hrs/day)")
    print(f"Total Effort:      {round(total_working_days, 2)} Working Days")
    print(f"Total Staff Hours: {round(total_hours, 1)} Hours")
    print(f"Calendar Duration: {pp_stats[0]} Yrs, {pp_stats[1]} Mos, {round(pp_stats[2], 1)} Days")
    print(f"{'='*50}\n")

    # --- VISUALIZATION ---
    chart_labels = [k for k, v in efforts.items() if v > 0]
    chart_values = [v for v in efforts.values() if v > 0]

    if chart_labels:
        plt.figure(figsize=(10, 4))
        colors = ['#4285F4', '#EA4335', '#FBBC05', '#34A853']
        bars = plt.bar(chart_labels, chart_values, color=colors[:len(chart_labels)])

        plt.title('Project Effort Breakdown (Working Days)', fontsize=14, pad=15)
        plt.ylabel('Days')
        plt.grid(axis='y', linestyle='--', alpha=0.5)

        # Add values on top of bars
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, round(yval, 1), ha='center', va='bottom')

        plt.show()